# Submission Q01: A with E30 uncertainty ranks 1–7 flipped (both edges)

| | |
|---|---|
| Submitted file | `Q01.csv` |
| Submitted (UTC) | 2026-10-08 00:12 |
| Public score | 0.96685 (175/181) |
| Rule | A with E30 uncertainty ranks 1–7 flipped (both edges) |

Added by E38 (`solution/portfolio_final.py`), which re-weighs the 25 final selections over three label models of the noisy band edges by how well each explains the known public scores; it filled the two slots left after P01–P20. Submitted as `Q01.csv` from the Kaggle notebook [kacchi-aloo-final-tickets](https://www.kaggle.com/code/hosen42/kacchi-aloo-final-tickets) v2 (source: `solution/kaggle_final_kernel/`).

**Run it:** on Kaggle, upload this notebook and add the competition data (*Add Input → Competitions → The Great Kacchi Aloo Mystery*), then *Save & Run All* and submit `submission.csv`. Locally, run it from this folder (it reads `../data/train.csv` and `../data/test.csv`). The last cell checks the output against the file that was actually submitted.

In [ ]:
import glob, hashlib, os, unicodedata
import numpy as np, pandas as pd

found = sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True)) + [p for p in ("../data/train.csv", "data/train.csv") if os.path.exists(p)]
root = os.path.dirname(found[0])
tr_raw = pd.read_csv(f"{root}/train.csv", dtype=str)
te_raw = pd.read_csv(f"{root}/test.csv", dtype=str)


def to_number(v):
    """Float from a cell; any Unicode decimal digit (e.g. Bangla) becomes ASCII."""
    if pd.isna(v):
        return np.nan
    s = "".join(str(unicodedata.decimal(c)) if unicodedata.decimal(c, None) is not None else c
                for c in str(v).strip().replace(",", ""))
    try:
        return float(s)
    except ValueError:
        return np.nan


def add_apg(d):
    """aloo per guest; the few aloo counts typed x10 (ratio > 3) are divided by 10."""
    guests, aloo = d.guests.map(to_number), d.aloo_count.map(to_number)
    aloo = np.where(aloo / guests > 3.0, aloo / 10, aloo)
    return d.assign(apg=aloo / guests)


tr = add_apg(tr_raw).dropna(subset=["apg"]).reset_index(drop=True)   # 24 train rows have no aloo_count
tr["y"] = tr.went_back_for_seconds.astype(int)
te = add_apg(te_raw)
print(f"data: {root} | train {len(tr)} rows with aloo_count | test {len(te)} rows")

In [ ]:
def midcut_cuts(r, y):
    """Each edge sits at the midpoint between neighbouring training ratios that maximises accuracy on its side of
    1.5 (median of tied optima)."""
    def best_cut(x, yy, side):
        xs = np.sort(np.unique(x))
        c = (xs[:-1] + xs[1:]) / 2
        accs = np.array([((x >= t) == yy).mean() if side == "lo" else ((x <= t) == yy).mean() for t in c])
        b = np.flatnonzero(accs == accs.max())
        return c[b[len(b) // 2]]
    m = r < 1.5
    return best_cut(r[m], y[m], "lo"), best_cut(r[~m], y[~m], "hi")


def band(x, lo, hi):
    return ((x >= lo) & (x <= hi)).astype(int)

In [ ]:
from sklearn.model_selection import StratifiedKFold


def sharp_band_probabilities(tr, te, bins=(0.0, 0.01, 0.035, 0.1, 0.4, np.inf), repeats=10, seed=0):
    """P(y=1) per test row = the sharp band's out-of-fold error rate at that distance from the nearest edge.
    Test rows inside the training gap around an edge get P from a uniform prior on where the cut lies."""
    r, y = tr.apg.values, tr.y.values
    lo, hi = midcut_cuts(r, y)
    d_oof, e_oof = [], []
    for k in range(repeats):
        for t, v in StratifiedKFold(5, shuffle=True, random_state=seed + k).split(tr, y):
            l2, h2 = midcut_cuts(r[t], y[t])
            d_oof.append(np.minimum(abs(r[v] - l2), abs(r[v] - h2)))
            e_oof.append(band(r[v], l2, h2) != y[v])
    d_oof, e_oof = np.concatenate(d_oof), np.concatenate(e_oof)
    idx = np.digitize(d_oof, bins[1:-1])
    err = np.array([(e_oof[idx == b].sum() + 0.5) / ((idx == b).sum() + 1.0) for b in range(len(bins) - 1)])
    below_lo, above_lo = r[r < lo].max(), r[(r >= lo) & (r < 1.5)].min()
    below_hi, above_hi = r[(r <= hi) & (r > 1.5)].max(), r[r > hi].min()
    x = te.apg.values
    d = np.minimum(abs(x - lo), abs(x - hi))
    e = err[np.digitize(d, bins[1:-1])]
    P = np.where(band(x, lo, hi) == 1, 1 - e, e)
    q_lo = np.clip((x - below_lo) / (above_lo - below_lo), 0, 1)
    q_hi = np.clip((above_hi - x) / (above_hi - below_hi), 0, 1)
    g_lo, g_hi = (x > below_lo) & (x < above_lo), (x > below_hi) & (x < above_hi)
    P = np.where(g_lo, q_lo * (1 - err[0]) + (1 - q_lo) * err[0], P)
    P = np.where(g_hi, q_hi * (1 - err[0]) + (1 - q_hi) * err[0], P)
    return P, (lo, hi), err


P, (LO, HI), err = sharp_band_probabilities(tr, te)
print(f"cut-points {LO:.5f} / {HI:.5f}; out-of-fold error rate by distance to the edge "
      f"(0-0.01, 0.01-0.035, 0.035-0.1, 0.1-0.4, >0.4): {np.round(err, 3)}")

In [ ]:
from scipy.special import betaln
from sklearn.isotonic import IsotonicRegression

r_tr, y_tr, x_te = tr.apg.values, tr.y.values.astype(float), te.apg.values


def plateaus(r, y, lo_win, hi_win):
    mid = (r > lo_win[1]) & (r < hi_win[0])
    out = (r < lo_win[0]) | (r > hi_win[1])
    jef = lambda m: (y[m].sum() + 0.5) / (m.sum() + 1.0)
    return jef(mid), jef(out)


def isotonic_edges(r, y, x, lo_win=(0.85, 1.15), hi_win=(1.80, 2.20)):
    """E30: increasing isotonic fit on the lower-edge window, decreasing on the upper, plateau rates elsewhere."""
    p_mid, p_out = plateaus(r, y, lo_win, hi_win)
    wl = (r >= lo_win[0]) & (r <= lo_win[1]); wh = (r >= hi_win[0]) & (r <= hi_win[1])
    iso_l = IsotonicRegression(increasing=True, y_min=0, y_max=1, out_of_bounds="clip").fit(r[wl], y[wl])
    iso_h = IsotonicRegression(increasing=False, y_min=0, y_max=1, out_of_bounds="clip").fit(r[wh], y[wh])
    p = np.where((x > lo_win[1]) & (x < hi_win[0]), p_mid, p_out)
    p = np.where((x >= lo_win[0]) & (x <= lo_win[1]), iso_l.predict(np.clip(x, *lo_win)), p)
    return np.where((x >= hi_win[0]) & (x <= hi_win[1]), iso_h.predict(np.clip(x, *hi_win)), p)


def changepoint_edge(r, y, win, x, a0=1.0, b0=1.0):
    """E31, one edge: cut uniform on the window, a constant rate on each side (Beta prior), cut integrated out."""
    m = (r >= win[0]) & (r <= win[1])
    rr, yy = r[m], y[m]
    u = np.unique(np.r_[win[0], rr, win[1]])
    lo_k, hi_k = u[:-1], u[1:]
    above = rr[None, :] >= hi_k[:, None]
    n_a, k_a = above.sum(1), (above * yy[None, :]).sum(1)
    n_b, k_b = len(rr) - n_a, yy.sum() - k_a
    ll = (betaln(a0 + k_a, b0 + n_a - k_a) + betaln(a0 + k_b, b0 + n_b - k_b) - 2 * betaln(a0, b0)
          + np.log(np.maximum(hi_k - lo_k, 1e-12)))
    post = np.exp(ll - ll.max()); post /= post.sum()
    th_a, th_b = (a0 + k_a) / (a0 + b0 + n_a), (a0 + k_b) / (a0 + b0 + n_b)
    q = np.clip((x[None, :] - lo_k[:, None]) / (hi_k - lo_k)[:, None], 0, 1)
    return post @ (q * th_a[:, None] + (1 - q) * th_b[:, None])


def changepoint_edges(r, y, x, lo_win=(0.85, 1.15), hi_win=(1.80, 2.25)):
    p_mid, p_out = plateaus(r, y, lo_win, hi_win)
    p = np.where((x > lo_win[1]) & (x < hi_win[0]), p_mid, p_out)
    il = (x >= lo_win[0]) & (x <= lo_win[1]); ih = (x >= hi_win[0]) & (x <= hi_win[1])
    p[il] = changepoint_edge(r, y, lo_win, x[il])
    p[ih] = changepoint_edge(r, y, hi_win, x[ih])
    return p


P30 = isotonic_edges(r_tr, y_tr, x_te)
P31 = changepoint_edges(r_tr, y_tr, x_te)
RANKING_P = {"E16": P, "E30": P30, "E31": P31}
print("rows with 0.25 < P < 0.75:", {k: int(((v > 0.25) & (v < 0.75)).sum()) for k, v in RANKING_P.items()})

In [ ]:
SIDE, START, WIDTH, RANKING = 'both', 0, 7, 'E30'   # ticket Q01: flip E30 ranks 1-7 (both edges)
A = band(te.apg.values, LO, HI)
order = np.lexsort((np.minimum(abs(te.apg.values - LO), abs(te.apg.values - HI)), np.abs(RANKING_P[RANKING] - 0.5)))
low = te.apg.values < 1.5
o = {"both": order, "lower": order[low[order]], "upper": order[~low[order]]}[SIDE]
pred = A.copy()
pred[o[START:START + WIDTH]] ^= 1
print(te.assign(P=RANKING_P[RANKING].round(3), A=A, ticket=pred).loc[A != pred, ["wedding_id", "guests", "aloo_count", "apg", "P", "A", "ticket"]]
      .sort_values("apg").to_string(index=False))

In [ ]:
sub = pd.DataFrame({"wedding_id": te.wedding_id, "went_back_for_seconds": np.asarray(pred).astype(int)})
sub.to_csv("submission.csv", index=False)
fp = hashlib.md5("".join(f"{i}{p}" for i, p in zip(sub.wedding_id, sub.went_back_for_seconds)).encode()).hexdigest()
print(f"wrote submission.csv: {sub.went_back_for_seconds.sum()} ones / {len(sub)} rows")
print(f"identical to the submitted file 'N0567.csv': {fp == '02cce4adcfe1dc9fe4dc99e3fd5021a4'}")

In [ ]:
assert fp == '02cce4adcfe1dc9fe4dc99e3fd5021a4'
PAIRS = [{'file': 'OPT01.csv', 'indices': [14, 205, 211, 352, 379], 'wedding_ids': ['W0815', 'W1006', 'W1012', 'W1153', 'W1180']}]
EXPECTED = {'OPT01.csv': '53369a484467854f7dda08ef6e37ef5f'}
for item in PAIRS:
    indices = item['indices']
    assert te.loc[indices, 'wedding_id'].tolist() == item['wedding_ids']
    probe=sub.copy()
    probe.loc[indices,'went_back_for_seconds'] = 1-probe.loc[indices,'went_back_for_seconds']
    assert (probe.went_back_for_seconds != sub.went_back_for_seconds).sum()==len(indices)
    check=hashlib.md5(''.join(f'{i}{p}' for i,p in zip(probe.wedding_id,probe.went_back_for_seconds)).encode()).hexdigest()
    assert check==EXPECTED[item['file']]
    probe.to_csv(item['file'],index=False)
    print(item['file'], item['wedding_ids'])
